In [ ]:
import os
import logging
from dotenv import load_dotenv, find_dotenv

import pandas as pd

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser

from core.flow.base import InferenceFlow
from core.flow.utils import compute_stat_metrics
from core.data.datasets import SlmFlowSyntheticDataset
from core.scheduler.base import SlmFlowScheduler
from core.scheduler.policies import ThresholdRoutingPolicy
from core.scheduler.features import (
    RerankingFeatureExtractor,
    ContextCompressionFeatureExtractor
)
from core.data.synthetic import (
    SyntheticDataGenerator,
    DatasetGenerationConfig
)
from core.io.models import (
    RagTaskOutput,
    SyntheticRowReranking,
    SyntheticRowCompression
)
from core.tasks.base import RagTask
from core.io.prompts import PROMPT_REGISTRY

import warnings
warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)

load_dotenv(find_dotenv())

# api keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
VLLM_API_KEY = os.getenv("VLLM_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# model selection
LLM_MODEL_NAME = os.getenv("LLM_MODEL_NAME")
SLM_MODEL_NAME = os.getenv("SLM_MODEL_NAME")
JUDGE_MODEL_NAME = os.getenv("JUDGE_MODEL_NAME")
GENERATOR_MODEL_NAME = os.getenv("GENERATOR_MODEL_NAME")

In [2]:
# dataset generation example

# configuring dataset structure (domains, task-types, amount of examples per batch)
config = DatasetGenerationConfig(
    tasks=["reranking", "context_compression"],
    domains=["coding", "history", "math", "medicine", "research"],
    difficulties=["easy", "medium", "complex"],
    batch_size=15
)

# setting up client, prompts and parsers
generator_llm_client = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model=GENERATOR_MODEL_NAME,
    max_tokens=4092,
    temperature=0.0,
)
generator_parsers = {
    "reranking": PydanticOutputParser(pydantic_object=SyntheticRowReranking),
    "context_compression": PydanticOutputParser(pydantic_object=SyntheticRowCompression),
}
generator_prompts = {
    "reranking": PROMPT_REGISTRY.reranking,
    "context_compression": PROMPT_REGISTRY.context_compression 
}

# instntiate the generator
generator = SyntheticDataGenerator(
    llm_api_client=generator_llm_client,
    config=config,
    parsers=generator_parsers,
    prompts=generator_prompts
)

In [ ]:
# generating the data with checkpoints on the disk
rows = await generator.generate_synthetic_dataset("generated/slm_flow_df_v2")

In [3]:
# experimental part

from langchain_core.rate_limiters import InMemoryRateLimiter

limiter = InMemoryRateLimiter(requests_per_second=3)

# loading data to the BaseDataset subclass
dataset = SlmFlowSyntheticDataset.from_files("generated/slm_flow_df")

# creating routing policies to each task (policies might be different)
policies_map = {
    # using statisical policies for example
    "reranking": ThresholdRoutingPolicy({
        "query_token_count":  ("lt",  500.0),
        "avg_lexical_overlap": ("gt", 0.1),
    }),
    "context_compression": ThresholdRoutingPolicy({
        "query_token_count":  ("lt",  500.0),
        "avg_lexical_overlap": ("gt", 0.1),
        "total_context_token_count": ("lt", 1000)
    })
}

# configuring the BaseFeatureExtractor subclasses for each task
extractors_map = {
    "reranking": RerankingFeatureExtractor(),
    "context_compression": ContextCompressionFeatureExtractor(),
}

# Some boilerplate code with api clients

common_params = {
    # for example, it might be openrouter
    "api_key": OPENROUTER_API_KEY,
    "base_url": "https://openrouter.ai/api/v1",
    "max_tokens": 4092,
    "temperature": 0.0,
    "rate_limiter": limiter
}
slm_client = ChatOpenAI(
    model=SLM_MODEL_NAME,
    **common_params
)
llm_client = ChatOpenAI(
    model=LLM_MODEL_NAME,
    **common_params
)
judge_llm = ChatOpenAI(
    model=JUDGE_MODEL_NAME,
    **common_params
)

# setting up scheduler, passing the policies and some language models clients to it
scheduler_dynamic = SlmFlowScheduler(
    policies=policies_map,
    extractors=extractors_map,
    llm=llm_client,
    slm=slm_client
)

# for ablation we can try to perform "clear-model" experiments
scheduler_slm = SlmFlowScheduler({}, extractors_map, llm_client, slm_client, mode="slm_only")
scheduler_llm = SlmFlowScheduler({}, extractors_map, llm_client, slm_client, mode="llm_only")

In [4]:
# define the Flow models

dynamic_flow = InferenceFlow(
    dataset=dataset[:30], # in example way, just define only 30 rows
    scheduler=scheduler_dynamic,
)
slm_flow = InferenceFlow(
    dataset=dataset[:30],
    scheduler=scheduler_slm,
)
llm_flow = InferenceFlow(
    dataset=dataset[:30],
    scheduler=scheduler_llm,
)

# and routine objects, like parsers, prompt templates and task-abstracts
tasks = {
    "reranking": RagTask,
    "context_compression": RagTask
}
parsers = {
    "reranking": PydanticOutputParser(pydantic_object=RagTaskOutput),
    "context_compression": PydanticOutputParser(pydantic_object=RagTaskOutput)
}
templates = {
    "reranking": PROMPT_REGISTRY.reranking_inference,
    "context_compression": PROMPT_REGISTRY.context_compression_inference,
}

In [ ]:
# now, we can run the experiment

# runtime: gpt 5.1 - 581.56 / qwen-3-235b-a22b - 1012.94
predictions = dynamic_flow.execute(
    tasks=tasks,
    parsers=parsers,
    templates=templates
)

In [ ]:
# evaluate the results
evaluation_results_dynamic = await dynamic_flow.evaluate_by_judge(judge_llm, PROMPT_REGISTRY.evaluation)
dump = pd.DataFrame([row.model_dump() for row in evaluation_results_dynamic])
dump.to_csv("generated/dyn_results_small.csv", index=False)

In [ ]:
# and compute some metrics

# reranking: {'bert_score_f1': 0.9076935589313507, 'judge_scores': 9.8}
# context compression: {'bert_score_f1': 0.31363720521330835, 'rouge_2_score': 0.16355011542426517, 'judge_scores': 9.65}
r_dynamic, c_dynamic = compute_stat_metrics(evaluation_results_dynamic)
print(r_dynamic, c_dynamic, sep="\n")

In [ ]:
# runtime: ministral-3b-2512 - 32.74
slm_only_predictions = slm_flow.execute(
    tasks=tasks,
    parsers=parsers,
    templates=templates
)

In [ ]:
evaluation_results_slm_only = await slm_flow.evaluate_by_judge(judge_llm, PROMPT_REGISTRY.evaluation)
dump = pd.DataFrame([row.model_dump() for row in evaluation_results_slm_only])
dump.to_csv("generated/slm_results_small.csv", index=False)

In [ ]:
# reranking: {'bert_score_f1': 0.5666903833548228, 'judge_scores': 7.9}
# context compression: {'bert_score_f1': 0.07902801294500629, 'rouge_2_score': 0.10926592279597426, 'judge_scores': 7.7833}

r_slm, c_slm = compute_stat_metrics(evaluation_results_slm_only)
print(r_slm, c_slm, sep="\n")

In [ ]:
# runtime: qwen-3-235b-a22b - 1008.45
llm_only_predictions = llm_flow.execute(
    tasks=tasks,
    parsers=parsers,
    templates=templates
)

In [ ]:
# reranking: {'bert_score_f1': 0.8958290, 'judge_scores': 9.9}
# context compression: {'bert_score_f1': 0.3048593, 'rouge_2_score': 0.1435922, 'judge_scores': 9.2}

evaluation_results_llm_only = await llm_flow.evaluate_by_judge(judge_llm, PROMPT_REGISTRY.evaluation)
dump = pd.DataFrame([row.model_dump() for row in evaluation_results_dynamic])
dump.to_csv("generated/llm_results_small.csv", index=False)

In [ ]:
r_llm, c_llm = compute_stat_metrics(evaluation_results_dynamic)
print(r_llm, c_llm, sep="\n")